In [1]:
path_course_annotations = './annotations/course_annotations.csv'
path_output_folder = './outputs/'

In [2]:
import pandas as pd
import os

annotations = pd.read_csv(path_course_annotations)

kc_cols = ["Syntax-Ontology_Expert", "Educational-Ontology_Expert", "AST_Parser", "Syntax-Ontology_LLM", "Educational-Ontology_LLM", "Free-Form_LLM"]

topic_activities = (
    annotations.groupby("topic")["activity"]
    .apply(list)
    .reset_index()
    .rename(columns={"activity": "exercises"})
)

annotations.head(3)

,topic,activity,Syntax-Ontology_Expert,Educational-Ontology_Expert,AST_Parser,Syntax-Ontology_LLM,Educational-Ontology_LLM,Free-Form_LLM
0,1,ps_hello,"FunctionCall,StringLiteral,print,ExpressionSta...","Printing,CallingLibraryFunction","Call,Str","BuiltInFunction,Call,ExpressionStatement,Funct...","CallingFunctionLibrary,CallingStandardFunction...","string operations,function call"
1,1,ps_python_addition,"FunctionCall,IntegerLiteral,print,SimpleAssign...","Addition,AssigningValueVariable,CallingLibrary...","Int,Add,Call,Assign","ArithmeticExpression,ArtithmeticOperator,Assig...","Addition,ArithmeticOperation,AssigningVariable...","function call,numerical types,variable manipul..."
2,1,ps_python_swap,"SimpleAssignmentStatement,ExpressionStatement,...","SwappingValueVariables,AssigningValueVariable",Assign,"AssignmentStatement,ExpressionStatement,Operat...","AssigningVariable,ManagingVariable,SwappingVar...","string operations,variable manipulation,functi..."


In [3]:
def build_kcs_dict(annotations, kc_col):
    result = {}
    for _, row in annotations.iterrows():
        kcs_str = row[kc_col]
        if pd.notna(kcs_str) and kcs_str != "":
            result[row["activity"]] = [k.strip() for k in kcs_str.split(",")]
        else:
            result[row["activity"]] = []
    return result

def collect_kcs(exercise_list, kcs_dict):
    all_kcs = set()
    for ex in exercise_list:
        all_kcs.update(kcs_dict.get(ex, []))
    return all_kcs

def place_activity(activity, partial_course, kcs_dict):
    activity_kcs = set(kcs_dict.get(activity, []))
    past_kcs = set()

    for _, row in partial_course.iterrows():
        current_kcs = collect_kcs(row["exercises"], kcs_dict)

        if activity_kcs.issubset(past_kcs | current_kcs):
            if activity_kcs & current_kcs:
                return (row["topic"], None)

        past_kcs.update(current_kcs)

    missing_kcs = list(activity_kcs - past_kcs)
    return (-1, missing_kcs)

In [4]:
os.makedirs(path_output_folder, exist_ok=True)

for kc_col in kc_cols:
    print(f"Running placement for: {kc_col}")
    kcs_dict = build_kcs_dict(annotations, kc_col)

    output_folder = os.path.join(path_output_folder, kc_col, "")
    os.makedirs(output_folder, exist_ok=True)

    placing_records = []
    missing_kcs_records = []

    for _, row in topic_activities.iterrows():
        topic_id = row["topic"]
        for exercise in row["exercises"]:
            topic_activities_copy = topic_activities.copy(deep=True)
            for i, temp_row in topic_activities_copy.iterrows():
                if exercise in temp_row["exercises"]:
                    topic_activities_copy.at[i, "exercises"] = [ex for ex in temp_row["exercises"] if ex != exercise]

            assigned_topic_id, missing_kcs = place_activity(exercise, topic_activities_copy, kcs_dict)

            placing_records.append({
                "exercise": exercise,
                "original_topic": topic_id,
                "assigned_topic": assigned_topic_id
            })

            if missing_kcs:
                missing_kcs_records.append({
                    "exercise": exercise,
                    "missing_kcs": missing_kcs
                })

    pd.DataFrame(placing_records).to_csv(output_folder + "placing_records.csv", sep="\t", index=False)
    pd.DataFrame(missing_kcs_records).to_csv(output_folder + "missing_kcs_records.csv", sep="\t", index=False)

    acc = sum(r["original_topic"] == r["assigned_topic"] for r in placing_records) / len(placing_records)
    print(f"  Accuracy: {acc:.3f} | Saved to {output_folder}\n")

print("Done.")

Running placement for: Syntax-Ontology_Expert
  Accuracy: 0.752 | Saved to ./outputs/Syntax-Ontology_Expert/

Running placement for: Educational-Ontology_Expert
  Accuracy: 0.789 | Saved to ./outputs/Educational-Ontology_Expert/

Running placement for: AST_Parser
  Accuracy: 0.731 | Saved to ./outputs/AST_Parser/

Running placement for: Syntax-Ontology_LLM
  Accuracy: 0.752 | Saved to ./outputs/Syntax-Ontology_LLM/

Running placement for: Educational-Ontology_LLM
  Accuracy: 0.690 | Saved to ./outputs/Educational-Ontology_LLM/

Running placement for: Free-Form_LLM
  Accuracy: 0.562 | Saved to ./outputs/Free-Form_LLM/

Done.
